# ── LLM-as-Judge (GPT-4o) for  0 noise neural models ──────────────

In [ ]:
# ── Cell 1: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install openai -q

In [ ]:
# ── Cell 10: LLM Judge ──
import openai, json, pandas as pd
from google.colab import userdata

client = openai.OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

SYSTEM_PROMPT = """You are a strict phoneme-sequence verifier for an Arabic text-to-phoneme parser.
Check ONLY whether the parser output matches the diacritised input using these rules.

PHONEME SET: b t θ dʒ ħ x d ð r z s ʃ sˤ dˤ tˤ ðˤ ʕ ɣ f q k l m n h w j ʔ a i u aː iː uː an in un

RULES:
1. SUN LETTERS (exactly these only): ت ث د ذ ر ز س ش ص ض ط ظ ل ن — ال assimilates before these only.
2. PAUSE FORM: ة is silent including its tanwin. Tanwin on other letters (an/in/un) is kept.
3. LONG VOWELS: ا after fatha=aː | ي after kasra=iː | و after damma=uː
4. SHADDA: consonant appears twice.
5. Transcribe only what diacritics explicitly mark. No inference.

NEVER flag as errors:
- Final case vowels (damma/kasra/fatha on last consonant) — optional in pause form
- Mid-word case vowels — optional
- وَ → w-a is correct (fatha on waw)
- وَا → w-aː is correct
- ها suffix → h-aː is correct
- Missing vowels on undiacritised consonants
- Final ت with sukun (ْ) — sukun means no vowel, consonant is not silent
- Tanwin on ة in pause form — ة is silent including its tanwin

Reply JSON only: {"accepted": true, "reason": ""} or {"accepted": false, "reason": "brief explanation of the error"}"""

def judge(word, phonemes):
    prompt = f"Arabic word: {word}\nParser output: {phonemes}\nEvaluate."
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user",   "content": prompt}],
        max_tokens=250, temperature=0
    )
    actual_model = resp.model
    text = resp.choices[0].message.content.strip()
    try:
        start = text.find('{')
        end   = text.rfind('}') + 1
        if start == -1 or end == 0:
            return False, f"no JSON: {text[:80]}", actual_model
        data = json.loads(text[start:end])
        return data.get("accepted", False), data.get("reason", ""), actual_model
    except Exception as e:
        return False, f"parse error: {e}: {text[:80]}", actual_model

# ── Run ──
test_df = pd.read_csv('/content/drive/MyDrive/Arabphon/test.csv')
sample  = test_df.sample(200, random_state=42)

results    = []
model_used = None

for _, row in sample.iterrows():
    accepted, reason, m = judge(row['word'], row['phonemes'])
    results.append({'word': row['word'], 'phonemes': row['phonemes'],
                    'accepted': accepted, 'reason': reason})
    if model_used is None:
        model_used = m

df = pd.DataFrame(results)
accepted_n = df['accepted'].sum()
rejected_n = len(df) - accepted_n

print(f"Actual model : {model_used}")
print(f"Accepted     : {accepted_n} ({100*accepted_n/len(df):.1f}%)")
print(f"Rejected     : {rejected_n} ({100*rejected_n/len(df):.1f}%)")

print(f"\n── Sample rejections (first 15) ──")
for _, r in df[~df['accepted']].head(15).iterrows():
    print(f"  {r['word']:<20} {r['phonemes']:<35} → {r['reason']}")

df.to_csv('/content/drive/MyDrive/Arabphon/llm_judge_results_v2.csv', index=False)
print("\nSaved.")